In [0]:
%sql
DROP TABLE IF EXISTS dev.bronze_db.customers;
DROP TABLE IF EXISTS dev.bronze_db.products;
DROP TABLE IF EXISTS dev.bronze_db.orders;
DROP TABLE IF EXISTS dev.silver_db.customers;
DROP TABLE IF EXISTS dev.silver_db.products;
DROP TABLE IF EXISTS dev.silver_db.orders;


In [0]:
# Import necessary modules
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, StringType, DateType

# Define the new Day 1 storage paths
customers_path = "/Volumes/datascience_catalog/default/day_1/Customers_*.parquet"
products_path = "/Volumes/datascience_catalog/default/day_1/Products_*.parquet"
orders_path = "/Volumes/datascience_catalog/default/day_1/Orders_*.parquet"

# Read Parquet files from Volume
df_customers = spark.read.parquet(customers_path)
df_products = spark.read.parquet(products_path)
df_orders = spark.read.parquet(orders_path)

# Convert data types for consistency
df_customers = df_customers.withColumn("Customer_ID", col("Customer_ID").cast(IntegerType())) \
                           .withColumn("start_date", col("start_date").cast(DateType()))

df_products = df_products.withColumn("start_date", col("start_date").cast(DateType()))

df_orders = df_orders.withColumn("Customer_ID", col("Customer_ID").cast(IntegerType())) \
                     .withColumn("start_date", col("start_date").cast(DateType()))

# Overwrite existing Bronze tables in Unity Catalog
df_customers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.bronze_db.customers")
df_products.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.bronze_db.products")
df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("dev.bronze_db.orders")

print("✅ Day 1 Bronze tables successfully overwritten!")


✅ Day 1 Bronze tables successfully overwritten!


In [0]:
%sql
-- Merge new customers into Silver Table
MERGE INTO dev.silver_db.customers AS target
USING (
  SELECT 
    1000 + ROW_NUMBER() OVER (ORDER BY Customer_ID) AS Customer_SK,
    Customer_ID,
    Customer_Name,
    start_date,
    DATE_FORMAT(CURRENT_TIMESTAMP, 'yyyy-MM-dd\'T\'HH:mm:ss') AS Load_date
  FROM dev.bronze_db.customers
) AS source
ON target.Customer_ID = source.Customer_ID
WHEN NOT MATCHED THEN 
  INSERT (Customer_SK, Customer_ID, Customer_Name, start_date, Load_date)
  VALUES (source.Customer_SK, source.Customer_ID, source.Customer_Name, source.start_date, source.Load_date);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
select * from dev.silver_db.customers

Customer_SK,Customer_ID,Customer_Name,start_date,Load_date
1001,1,Alice,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1002,2,Bob,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1003,3,Charlie,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1004,4,David,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1005,5,Ella,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1006,6,Frank,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1007,7,Grace,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1008,8,Hannah,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1009,9,Ian,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z
1010,10,Jane,2025-03-29T00:00:00.000Z,2025-03-30T18:37:32.000Z


In [0]:
%sql
-- Merge new products into Silver Table
MERGE INTO dev.silver_db.products AS target
USING (
  SELECT 
    2000 + ROW_NUMBER() OVER (ORDER BY Product_ID) AS Product_SK,
    Product_ID,
    Product_Name,
    start_date,
    DATE_FORMAT(CURRENT_TIMESTAMP, 'yyyy-MM-dd\'T\'HH:mm:ss') AS Load_date
  FROM dev.bronze_db.products
) AS source
ON target.Product_ID = source.Product_ID
WHEN NOT MATCHED THEN 
  INSERT (Product_SK, Product_ID, Product_Name, start_date, Load_date)
  VALUES (source.Product_SK, source.Product_ID, source.Product_Name, source.start_date, source.Load_date);


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
select * from dev.silver_db.products

Product_SK,Product_ID,Product_Name,start_date,Load_date
2001,P101,Laptop,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2002,P102,Smartphone,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2003,P103,Tablet,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2004,P104,Monitor,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2005,P105,Headphones,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2006,P106,Keyboard,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2007,P107,Mouse,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2008,P108,Printer,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2009,P109,Speaker,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z
2010,P110,Webcam,2025-03-13T00:00:00.000Z,2025-03-21T06:42:31.000Z


In [0]:
%sql
-- Step 1: Mark Previous Orders as Inactive When Product Changes
MERGE INTO dev.silver_db.orders AS prev
USING (
  SELECT o.Order_ID, c.Customer_SK, p.Product_SK
  FROM dev.bronze_db.orders o
  JOIN dev.silver_db.customers c ON o.Customer_ID = c.Customer_ID
  JOIN dev.silver_db.products p ON o.Product_ID = p.Product_ID
) AS new
ON prev.Order_ID = new.Order_ID
  AND prev.Customer_SK = new.Customer_SK
  AND prev.Product_SK <> new.Product_SK
WHEN MATCHED THEN 
  UPDATE SET prev.End_date = DATE_FORMAT(CURRENT_TIMESTAMP, 'yyyy-MM-dd\'T\'HH:mm:ss');

-- Step 2: Insert New or Updated Orders
INSERT INTO dev.silver_db.orders
SELECT 
  o.Order_ID,
  c.Customer_SK,
  p.Product_SK,
  o.start_date,
  DATE_FORMAT(CURRENT_TIMESTAMP, 'yyyy-MM-dd\'T\'HH:mm:ss') AS Load_date,
  DATE_FORMAT('2999-12-31 23:59:59', 'yyyy-MM-dd\'T\'HH:mm:ss') AS End_date
FROM dev.bronze_db.orders o
JOIN dev.silver_db.customers c ON o.Customer_ID = c.Customer_ID
JOIN dev.silver_db.products p ON o.Product_ID = p.Product_ID
WHERE NOT EXISTS (
  SELECT 1 FROM dev.silver_db.orders s
  WHERE s.Order_ID = o.Order_ID 
    AND s.Customer_SK = c.Customer_SK 
    AND s.Product_SK = p.Product_SK
);

-- Step 3: Handle Missing Orders (Mark as Deleted)
UPDATE dev.silver_db.orders
SET End_date = Load_date
WHERE Order_ID NOT IN (SELECT Order_ID FROM dev.bronze_db.orders);


num_affected_rows
1


In [0]:
%sql
select * from dev.silver_db.orders

Order_ID,Customer_SK,Product_SK,start_date,Load_date,End_date
O105,1005,2005,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2025-03-30T18:37:47.000Z
O101,1001,2001,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2999-12-31T23:59:59.000Z
O102,1002,2002,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2999-12-31T23:59:59.000Z
O103,1003,2003,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2999-12-31T23:59:59.000Z
O104,1004,2004,2025-03-29T00:00:00.000Z,2025-03-30T18:37:47.000Z,2025-03-30T18:38:57.000Z
O104,1004,2007,2025-03-29T00:00:00.000Z,2025-03-30T18:39:01.000Z,2999-12-31T23:59:59.000Z
O106,1006,2006,2025-03-29T00:00:00.000Z,2025-03-30T18:39:01.000Z,2999-12-31T23:59:59.000Z
O107,1007,2007,2025-03-29T00:00:00.000Z,2025-03-30T18:39:01.000Z,2999-12-31T23:59:59.000Z
